# PyNeuTube Tutorial

A compact notebook for the public workflow.
The focus is on building a reusable personal tracing workflow with stage storage, visualization, and optional config overrides.

## Install

```bash
python -m pip install pyneutube
# or
conda install -c conda-forge pyneutube
```

Latest source:

```bash
python -m pip install "git+https://github.com/YunZhixi98/pyNeuTube.git"
```

In [1]:
from pathlib import Path

from pyneutube import (
    ImageParser,
    connect_trace_chains,
    extract_trace_seeds,
    generate_trace_chains,
    load_trace_stage,
    preprocess_volume,
    save_chain_overlay_figure,
    save_overlay_figure,
    save_seed_overlay_figure,
    trace_file,
)

reference_image = Path("../" + "examples/data/reference_volume.nii.gz")
artifact_dir = Path("notebook_artifacts")
artifact_dir.mkdir(exist_ok=True)
artifact_dir

WindowsPath('notebook_artifacts')

## Quick single-file trace

If `visualization_dir` is set, PyNeuTube writes:

- `result/` for the final reconstruction overlay
- `seeds/` for scored tracing seeds
- `chains/` for traced chains

In [2]:
result = trace_file(
    reference_image,
    output_swc=artifact_dir / "reference_trace.swc",
    visualization_dir=artifact_dir / "visualizations",
    n_jobs=1,
    timeout=600,
    verbose=1,
    overwrite=True,
)

print(result.threshold, len(result.seeds), len(result.chains))
print(result.output_swc)
print(result.output_seed_visualization)
print(result.output_chain_visualization)
print(result.output_visualization)

Loading ..\examples\data\reference_volume.nii.gz as nifti
load_image: 0.154s
subtract_background: 1.766s
local_max_filter: 4.104s
triangle_threshold=8.000: 0.003s
refine_local_max_threshold=8.000: 0.133s
binary_mask: 5.485s
3169 seeds found
--> seed_init: 3.164730s


Scoring seeds: 100%|██████████| 3169/3169 [01:07<00:00, 46.64seed/s] 


Number of labeled seeds: 841
--> seed_score: 71.133644s
Number of seed after filtering: 2273
generate_tracing_seeds: 71.138s


Generating chains: 100%|██████████| 2273/2273 [02:48<00:00, 13.51it/s]


Number of chains: 1238
Number of chains after filtering: 808
generate_neuron_trace: 168.320s
prepare_chain_conn: graph vertices=808, edges=1655
remove_redundant_edges: graph vertices=808, edges=1322
circles_to_tree: graph nodes=8184, edges=8694
minimum_spanning_tree: nodes=8184, edges=8042
--> from_graph [length=8184]: 2.34009
--> remove_zigzag [length=7840]: 1.65182
--> tune_branch [length=7840]: 2.10262
--> remove_spur [length=7688]: 1.83880
--> merge close point [length=7676]: 2.00485
--> remove overshoot [length=7479]: 2.03061
--> downsample [length=5670]: 3.87891
Total 142 subtrees, Removed 135 subtrees, Remove mean length: 275.1746285035057
--> remove subtrees by length [length=4665]: 0.95354
reconstruct: 34.589s
SWC data saved to notebook_artifacts\reference_trace.swc
8.0 2273 808
notebook_artifacts\reference_trace.swc
notebook_artifacts\visualizations\seeds\reference_volume.nii.gz.png
notebook_artifacts\visualizations\chains\reference_volume.nii.gz.png
notebook_artifacts\visual

## Build a personal staged workflow

`preprocess_volume(..., output_path=...)` stores only lightweight threshold metadata.
The downstream stages therefore take the original image again instead of reading a large cached signal volume.

In [3]:
image = ImageParser(reference_image).load()

signal_image, binary_image, threshold = preprocess_volume(
    image,
    verbose=1,
    output_path=artifact_dir / "preprocess.pkl",
)

print(signal_image.shape, binary_image.shape, threshold)

subtract_background: 2.374s
local_max_filter: 5.391s
triangle_threshold=8.000: 0.003s
refine_local_max_threshold=8.000: 0.132s
binary_mask: 6.786s
(256, 512, 512) (256, 512, 512) 8.0


In [4]:
preprocessed = load_trace_stage(artifact_dir / "preprocess.pkl")
print(preprocessed.threshold)

8.0


In [ ]:
seeds = extract_trace_seeds(
    image,
    threshold=preprocessed.threshold,
    n_jobs=1,
    verbose=1,
    output_path=artifact_dir / "seeds.pkl",
    visualization_path=artifact_dir / "visualizations" / "seeds" / "staged_run.png",
)

print(len(seeds))

subtract_background: 1.780s
binary_mask: 5.929s
3169 seeds found
--> seed_init: 1.934496s


Scoring seeds: 100%|██████████| 3169/3169 [00:35<00:00, 88.11seed/s] 


Number of labeled seeds: 841
--> seed_score: 38.676314s
Number of seed after filtering: 2273
2273


In [6]:
chains = generate_trace_chains(
    seeds,
    image,
    max_seeds=None,
    filter_chains=True,
    verbose=1,
    output_path=artifact_dir / "chains.pkl",
    visualization_path=artifact_dir / "visualizations" / "chains" / "staged_run.png",
)

print(len(chains))

subtract_background: 2.100s


Generating chains: 100%|██████████| 2273/2273 [03:58<00:00,  9.55it/s]


Number of chains: 1238
Number of chains after filtering: 808
808


In [7]:
chains = load_trace_stage(artifact_dir / "chains.pkl")

neuron = connect_trace_chains(
    chains,
    image,
    verbose=1,
    visualization_path=artifact_dir / "visualizations" / "result" / "staged_run.png",
)

neuron.save_swc(artifact_dir / "staged_trace.swc")
print(len(neuron))

subtract_background: 2.137s
prepare_chain_conn: graph vertices=808, edges=1655
remove_redundant_edges: graph vertices=808, edges=1322
circles_to_tree: graph nodes=8184, edges=8694
minimum_spanning_tree: nodes=8184, edges=8042
--> from_graph [length=8184]: 2.67995
--> remove_zigzag [length=7840]: 2.34613
--> tune_branch [length=7840]: 2.45992
--> remove_spur [length=7688]: 2.25382
--> merge close point [length=7676]: 2.29855
--> remove overshoot [length=7479]: 2.07430
--> downsample [length=5670]: 4.01870
Total 142 subtrees, Removed 135 subtrees, Remove mean length: 275.1746285035057
--> remove subtrees by length [length=4665]: 0.86168
SWC data saved to notebook_artifacts\staged_trace.swc
4665


### Threshold and binary-image control

For `extract_trace_seeds(...)`:

- if `binary_image` is provided, it is used directly
- otherwise, if `threshold` is provided, binary thresholding is rebuilt internally
- otherwise, the full preprocess path is run internally

## Config override

By default, the built-in tracer config is used.
To test a custom config module, pass its import path through `config=`.

In [8]:
# Example only. Replace with your own module path.
# result = trace_file(
#     reference_image,
#     output_swc=artifact_dir / "reference_trace_custom.swc",
#     visualization_dir=artifact_dir / "visualizations_custom",
#     config="my_project.pyneutube_config",
#     overwrite=True,
# )

## Manual visualization helpers

The three overlay helpers can also be called directly in custom workflows.

In [9]:
save_seed_overlay_figure(
    image,
    seeds,
    artifact_dir / "manual_seed_overlay.png",
)

save_chain_overlay_figure(
    image,
    chains,
    artifact_dir / "manual_chain_overlay.png",
)

save_overlay_figure(
    image,
    neuron,
    artifact_dir / "manual_result_overlay.png",
)

WindowsPath('notebook_artifacts/manual_result_overlay.png')